## Preparacion iniciales con la base de datos


In [2]:
# Importar las librerías necesarias
import pandas as pd
from pymongo import MongoClient
import numpy as np

# --- Conexión a MongoDB ---
# Por defecto, MongoDB corre en el puerto 27017 pero yo la cambié a 27018
client = MongoClient('mongodb://localhost:27018/')

# --- Seleccionar la Base de Datos y la Colección ---
# Si la base de datos o la colección no existen, MongoDB las creará automáticamente.
db = client['calidad_aire_db']
collection = db['mediciones']

print("¡Conexión a MongoDB exitosa!")
print("Bases de datos existentes:", client.list_database_names())

¡Conexión a MongoDB exitosa!
Bases de datos existentes: ['admin', 'config', 'local']


## Cargar y Procesar el Archivo CSV con Pandas 


In [10]:
# --- PASO de cargar y procesar CORREGIDO: Cargar, LIMPIAR y Procesar el CSV ---

ruta_archivo = 'consolidado_schema_4.csv'
df = pd.read_csv(ruta_archivo)

# --- INICIO DE LA LIMPIEZA ---

# 1. Limpiar espacios en los nombres de las columnas
df.columns = df.columns.str.strip()

# 2. Limpiar espacios en todas las celdas que sean de tipo texto
for col in df.select_dtypes(['object']):
    df[col] = df[col].str.strip()

# --- FIN DE LA LIMPIEZA ---


# --- Continuación del procesamiento ---

# 3. Convertir la columna 'date' a formato de fecha y hora
df['date'] = pd.to_datetime(df['date'])

# 4. Convertir columnas numéricas de texto a números reales
columnas_numericas = ['pm25', 'pm10', 'o3', 'no2', 'so2', 'co']
for col in columnas_numericas:
    df[col] = pd.to_numeric(df[col], errors='coerce') # 'coerce' convierte errores en NaT/NaN

# 5. Reemplazar valores nulos (NaN) con None para MongoDB
df = df.replace({np.nan: None})

# 6. Convertir el DataFrame a una lista de diccionarios
data_to_insert = df.to_dict('records')

# Mostrar los primeros 5 registros para verificar la limpieza
print(f"Se han cargado y procesado {len(data_to_insert)} registros.")
print("Ejemplo de un registro LIMPIO a insertar:")
print(data_to_insert[0])

Se han cargado y procesado 188626 registros.
Ejemplo de un registro LIMPIO a insertar:
{'date': Timestamp('2025-08-05 00:00:00'), 'pm25': 20.0, 'pm10': 33.0, 'o3': 32.0, 'no2': 8.0, 'so2': 4.0, 'co': 5.0, 'estado': 'guanajuato', 'fuente_archivo': 'celaya-policia, celaya-air-quality.csv'}


## Insertar los Datos en MongoDB

In [11]:
# --- Insertar los Datos ---

# (Opcional pero recomendado) Borrar datos existentes en la colección para evitar duplicados si corro el script varias veces.
collection.delete_many({})

# Insertar la lista de diccionarios en la colección
result = collection.insert_many(data_to_insert)

print(f"¡Éxito! Se han insertado {len(result.inserted_ids)} documentos en la colección 'mediciones'.")

¡Éxito! Se han insertado 188626 documentos en la colección 'mediciones'.


## Consultas a la Base de Datos

#### Consulta 1: Obtener un solo documento

In [12]:
# find_one() devuelve el primer documento que encuentra
un_documento = collection.find_one()
print(un_documento)

{'_id': ObjectId('68bb5da58f1a50659fa3fca4'), 'date': datetime.datetime(2025, 8, 5, 0, 0), 'pm25': 20.0, 'pm10': 33.0, 'o3': 32.0, 'no2': 8.0, 'so2': 4.0, 'co': 5.0, 'estado': 'guanajuato', 'fuente_archivo': 'celaya-policia, celaya-air-quality.csv'}


#### Consulta 2: Buscar todos los documentos de un 'estado' específico

In [22]:
query_estado = {'estado': 'san_luis_potosi'} 

# find() devuelve un "cursor", que es un objeto iterable.
documentos_estado = collection.find(query_estado)

# Iteramos sobre el cursor para imprimir los resultados
for doc in documentos_estado:
    print(doc)

{'_id': ObjectId('68bb5da58f1a50659fa4669d'), 'date': datetime.datetime(2025, 7, 2, 0, 0), 'pm25': 29.0, 'pm10': 13.0, 'o3': 13.0, 'no2': 5.0, 'so2': None, 'co': 2.0, 'estado': 'san_luis_potosi', 'fuente_archivo': 'industriales-potosinos asociados, san luis potosi estatal, san luis potosi-air-quality.csv'}
{'_id': ObjectId('68bb5da58f1a50659fa4669e'), 'date': datetime.datetime(2025, 7, 3, 0, 0), 'pm25': 22.0, 'pm10': 21.0, 'o3': 16.0, 'no2': 6.0, 'so2': 2.0, 'co': 3.0, 'estado': 'san_luis_potosi', 'fuente_archivo': 'industriales-potosinos asociados, san luis potosi estatal, san luis potosi-air-quality.csv'}
{'_id': ObjectId('68bb5da58f1a50659fa4669f'), 'date': datetime.datetime(2025, 7, 4, 0, 0), 'pm25': 43.0, 'pm10': 25.0, 'o3': 18.0, 'no2': 6.0, 'so2': 1.0, 'co': 2.0, 'estado': 'san_luis_potosi', 'fuente_archivo': 'industriales-potosinos asociados, san luis potosi estatal, san luis potosi-air-quality.csv'}
{'_id': ObjectId('68bb5da58f1a50659fa466a0'), 'date': datetime.datetime(2025, 

#### Consulta 3: Buscar con condiciones numéricas (mayor que, menor que)

In [21]:
# Operadores de consulta de MongoDB:
# $gt: greater than (mayor que)
# $lt: less than (menor que)
# $gte: greater than or equal (mayor o igual que)
# $lte: less than or equal (menor o igual que)

# Buscar mediciones con pm25 > 850
query_pm25_alto = {'pm25': {'$gt': 850}}

resultados_altos = collection.find(query_pm25_alto)

for doc in resultados_altos:
    print(f"Fecha: {doc['date']}, Estado: {doc['estado']}, PM2.5: {doc['pm25']}")

Fecha: 2024-03-21 00:00:00, Estado: san_luis_potosi, PM2.5: 999.0
Fecha: 2024-02-14 00:00:00, Estado: coahuila, PM2.5: 999.0
Fecha: 2018-04-13 00:00:00, Estado: coahuila, PM2.5: 939.0
Fecha: 2018-04-14 00:00:00, Estado: coahuila, PM2.5: 892.0
Fecha: 2018-09-12 00:00:00, Estado: aguascalientes, PM2.5: 853.0
Fecha: 2017-11-05 00:00:00, Estado: aguascalientes, PM2.5: 860.0


#### Consulta 4: Consulta compuesta (múltiples condiciones)

In [31]:
# Condiciones: estado de ciudad_de_mexico Y pm10 > 100
query_compuesta = {
    'estado': 'cdmx',
    'pm10': {'$gt': 100}
}

resultados_compuestos = collection.find(query_compuesta)

#for doc in resultados_compuestos:
    #print(doc)

#resultados con formato bonito
for doc in resultados_compuestos:
    print(f"Fecha: {doc['date']}, Estado: {doc['estado']}, PM10: {doc['pm10']}")

Fecha: 2019-11-23 00:00:00, Estado: cdmx, PM10: 110.0
Fecha: 2021-03-28 00:00:00, Estado: cdmx, PM10: 115.0
Fecha: 2025-03-21 00:00:00, Estado: cdmx, PM10: 105.0


#### Consulta 5: Proyección (seleccionar solo algunas columnas(campos))

In [40]:
# Obtener solo la fecha, el estado y el valor de o3 para mediciones con o3 > 250.0
# El 1 indica que queremos incluir el campo, el 0 que lo queremos excluir.
# El _id se incluye por defecto, por eso lo excluimos explícitamente con 0.
query = {'o3': {'$gt': 250.0}}
proyeccion = {'date': 1, 'estado': 1, 'o3': 1, '_id': 0}

resultados_proyectados = collection.find(query, proyeccion)

#for doc in resultados_proyectados:
 #   print(doc)

#resultados con formato bonito
for doc in resultados_proyectados:
    print(f"Fecha: {doc['date']}, Estado: {doc['estado']}, O3: {doc['o3']}")

Fecha: 2017-05-10 00:00:00, Estado: guanajuato, O3: 500.0
Fecha: 2017-05-11 00:00:00, Estado: guanajuato, O3: 500.0
Fecha: 2017-05-24 00:00:00, Estado: guanajuato, O3: 500.0
Fecha: 2017-05-29 00:00:00, Estado: guanajuato, O3: 500.0
Fecha: 2017-05-30 00:00:00, Estado: guanajuato, O3: 500.0
Fecha: 2017-05-09 00:00:00, Estado: guanajuato, O3: 500.0
Fecha: 2017-05-23 00:00:00, Estado: guanajuato, O3: 500.0
Fecha: 2017-05-28 00:00:00, Estado: guanajuato, O3: 500.0
Fecha: 2017-05-31 00:00:00, Estado: guanajuato, O3: 500.0
Fecha: 2020-01-20 00:00:00, Estado: guanajuato, O3: 251.0
Fecha: 2017-05-31 00:00:00, Estado: guanajuato, O3: 500.0
Fecha: 2017-05-12 00:00:00, Estado: guanajuato, O3: 447.0
Fecha: 2017-05-24 00:00:00, Estado: guanajuato, O3: 388.0
Fecha: 2017-05-25 00:00:00, Estado: guanajuato, O3: 500.0
Fecha: 2017-05-29 00:00:00, Estado: guanajuato, O3: 500.0
Fecha: 2017-05-30 00:00:00, Estado: guanajuato, O3: 500.0
Fecha: 2017-05-31 00:00:00, Estado: guanajuato, O3: 500.0
Fecha: 2021-12

#### Consulta 6: Contar documentos que cumplen una condición

In [43]:
# Contar cuántas mediciones tienen un nivel de so2 > 200
query_so2 = {'so2': {'$gt': 200}}
conteo = collection.count_documents(query_so2)

print(f"Hay {conteo} mediciones con un nivel de SO2 superior a 200.")

Hay 166 mediciones con un nivel de SO2 superior a 200.


#### Consulta 7 (Avanzada): Agregación

Las agregaciones me permiten realizar operaciones complejas como agrupar y calcular promedios, sumas, etc.

In [44]:
# Calcular el promedio de 'pm25' para cada 'estado'.
pipeline = [
    {
        '$group': {
            '_id': '$estado',  # Agrupar por el campo 'estado'
            'promedio_pm25': {'$avg': '$pm25'} # Calcular el promedio del campo 'pm25'
        }
    },
    {
        '$sort': {'promedio_pm25': -1} # Ordenar de mayor a menor promedio
    }
]

resultados_agregados = collection.aggregate(pipeline)

print("Promedio de PM2.5 por estado:")
for resultado in resultados_agregados:
    print(f"Estado: {resultado['_id']}, Promedio PM2.5: {resultado['promedio_pm25']:.2f}")

Promedio de PM2.5 por estado:
Estado: aguascalientes, Promedio PM2.5: 113.06
Estado: coahuila, Promedio PM2.5: 103.75
Estado: puebla, Promedio PM2.5: 91.28
Estado: michoacan, Promedio PM2.5: 77.03
Estado: estado_de_mexico, Promedio PM2.5: 76.48
Estado: jalisco, Promedio PM2.5: 75.54
Estado: durango, Promedio PM2.5: 71.89
Estado: san_luis_potosi, Promedio PM2.5: 71.52
Estado: morelos, Promedio PM2.5: 71.22
Estado: nayarit, Promedio PM2.5: 70.14
Estado: cdmx, Promedio PM2.5: 65.67
Estado: guanajuato, Promedio PM2.5: 64.39
Estado: hidalgo, Promedio PM2.5: 61.94
Estado: monterrey, Promedio PM2.5: 56.91
Estado: tlaxcala, Promedio PM2.5: 55.20
Estado: chihuahua, Promedio PM2.5: 50.83
